# 🚀 Модуль 8. Урок 3. Промпты для генерации SQL и Python кода: контекст схемы базы данных

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_08_prompt_engineering/lesson_03_prompting_for_sql_and_python.ipynb)

## 📌 Бизнес-контекст и методология
Чтобы LLM сгенерировала корректный SQL-запрос, ей необходим точный DDL-контекст таблиц и связей.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Формирование системного контекста схемы данных
schema_context = """
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    order_date DATE,
    status VARCHAR(20) -- ['Completed', 'Refunded', 'Cancelled']
);

CREATE TABLE order_items (
    order_item_id INT PRIMARY KEY,
    order_id INT REFERENCES orders(order_id),
    total_amount DECIMAL(10, 2)
);
"""

user_question = "Посчитай выручку по завершенным заказам за 2026 год"
system_instruction = f"Используй схему:\n{schema_context}\nНапиши чистый PostgreSQL запрос без пояснений."

print(system_instruction[:250] + "...")


Используй схему:

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    order_date DATE,
    status VARCHAR(20) -- ['Completed', 'Refunded', 'Cancelled']
);

CREATE TABLE order_items (
    order_item_id INT PRIMARY KEY,
    ord...


### 🎯 Практическое задание 1: Проверка наличия таблиц в схеме
Убедитесь, что обе таблицы `orders` и `order_items` переданы в контекст.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert "CREATE TABLE orders" in schema_context
assert "CREATE TABLE order_items" in schema_context
print("✅ Тест успешно пройден: контекст схемы полон!")


✅ Тест успешно пройден: контекст схемы полон!


### 💡 Семантическая ловушка
В реальной базе колонка может называться `order_items.total_amount`, а не `orders.amount`. Без передачи DDL модель почти наверняка ошибется в имени колонки.